In [1]:
import db_queries
from get_draws.api import get_draws
import pandas as pd, numpy as np

In [2]:
locations = ["india", "nigeria"]
index_cols = ["location", "sex", "age_start", "age_end", "vehicle", "wealth_quintile"]

age_group_ids = [
    2,3,
    388,389,
    6,7,8,9,10,11,12,13,14,15,16,17,18,19,20, 30, 31, 32, 235
]
sex_ids = [1,2]

DRAWS = [f'draw_{i}' for i in range(500)] # NOTE: Some GBD 2021 things return 1,000 but others don't

In [3]:
from vivarium_inputs import utility_data

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [4]:
location_crosswalk = pd.DataFrame({
    'location': [location.title() for location in locations],
    'location_id': [utility_data.get_location_id(location.title()) for location in locations],
})
location_crosswalk

,location,location_id
0,India,163
1,Nigeria,214


In [5]:
sex_crosswalk = pd.DataFrame({
    'sex': ['Male', 'Female'],
    'sex_id': [1, 2],
})
sex_crosswalk

,sex,sex_id
0,Male,1
1,Female,2


In [6]:
mean_difference = (
    pd.read_csv('../0100_data_prep/results/iron/fortification_hemoglobin_effects.csv')
        .rename(columns={'vehicle_name': 'vehicle'})
        .set_index('vehicle').value
)
mean_difference

vehicle
rice        3.25
salt        4.40
bouillon    4.20
Name: value, dtype: float64

In [7]:
effective_baseline_coverage = pd.concat([
    pd.read_csv(f'../0100_data_prep/results/iron/effective_baseline_coverage/{location}.csv')
        .assign(location=location.title())
        .rename(columns={'vehicle_name': 'vehicle'})
    for location in locations
])
effective_baseline_coverage

,sex,age_start,age_end,wealth_quintile,vehicle,value,location
0,Female,0.0,5.0,lowest,rice,0.294750,India
1,Female,0.0,5.0,second,rice,0.298731,India
2,Female,0.0,5.0,middle,rice,0.278976,India
3,Female,0.0,5.0,fourth,rice,0.239271,India
4,Female,0.0,5.0,highest,rice,0.137355,India
5,Female,5.0,15.0,lowest,rice,0.384923,India
6,Female,5.0,15.0,second,rice,0.372437,India
7,Female,5.0,15.0,middle,rice,0.344120,India
8,Female,5.0,15.0,fourth,rice,0.298223,India
9,Female,5.0,15.0,highest,rice,0.170117,India


In [8]:
def expand(df):
    for col in sorted(list(set(df.columns) - {'value'})):
        if df[col].isnull().any():
            df = pd.concat([
                df[df[col].notnull()],
                *[df[df[col].isnull()].assign(**{col: value}) for value in df[df[col].notnull()][col].unique()]
            ])
    
    return df

In [9]:
effective_baseline_coverage["age_start"] = effective_baseline_coverage.age_start.fillna(0)
effective_baseline_coverage["age_end"] = effective_baseline_coverage.age_end.fillna(100)

In [10]:
effective_baseline_coverage = expand(effective_baseline_coverage)
effective_baseline_coverage

,sex,age_start,age_end,wealth_quintile,vehicle,value,location
0,Female,0.0,5.0,lowest,rice,0.294750,India
1,Female,0.0,5.0,second,rice,0.298731,India
2,Female,0.0,5.0,middle,rice,0.278976,India
3,Female,0.0,5.0,fourth,rice,0.239271,India
4,Female,0.0,5.0,highest,rice,0.137355,India
5,Female,5.0,15.0,lowest,rice,0.384923,India
6,Female,5.0,15.0,second,rice,0.372437,India
7,Female,5.0,15.0,middle,rice,0.344120,India
8,Female,5.0,15.0,fourth,rice,0.298223,India
9,Female,5.0,15.0,highest,rice,0.170117,India


In [11]:
effective_counterfactual_coverage = pd.concat([
    pd.read_csv(f'../0100_data_prep/results/iron/intervention/effective_intervention_coverage/{location}.csv')
        .assign(location=location.title())
        .rename(columns={'vehicle_name': 'vehicle'})
    for location in locations
])
effective_counterfactual_coverage

,sex,age_start,age_end,wealth_quintile,vehicle,value,location
0,Female,0.0,5.0,lowest,rice,0.480406,India
1,Female,0.0,5.0,second,rice,0.489626,India
2,Female,0.0,5.0,middle,rice,0.485256,India
3,Female,0.0,5.0,fourth,rice,0.465103,India
4,Female,0.0,5.0,highest,rice,0.414788,India
5,Female,5.0,15.0,lowest,rice,0.518084,India
6,Female,5.0,15.0,second,rice,0.516194,India
7,Female,5.0,15.0,middle,rice,0.503733,India
8,Female,5.0,15.0,fourth,rice,0.480683,India
9,Female,5.0,15.0,highest,rice,0.422328,India


In [12]:
effective_counterfactual_coverage["age_start"] = effective_counterfactual_coverage.age_start.fillna(0)
effective_counterfactual_coverage["age_end"] = effective_counterfactual_coverage.age_end.fillna(100)

In [13]:
effective_counterfactual_coverage = expand(effective_counterfactual_coverage)
effective_counterfactual_coverage

,sex,age_start,age_end,wealth_quintile,vehicle,value,location
0,Female,0.0,5.0,lowest,rice,0.480406,India
1,Female,0.0,5.0,second,rice,0.489626,India
2,Female,0.0,5.0,middle,rice,0.485256,India
3,Female,0.0,5.0,fourth,rice,0.465103,India
4,Female,0.0,5.0,highest,rice,0.414788,India
5,Female,5.0,15.0,lowest,rice,0.518084,India
6,Female,5.0,15.0,second,rice,0.516194,India
7,Female,5.0,15.0,middle,rice,0.503733,India
8,Female,5.0,15.0,fourth,rice,0.480683,India
9,Female,5.0,15.0,highest,rice,0.422328,India


In [14]:
age_groups = db_queries.get_age_metadata(release_id=9).set_index('age_group_id').rename(columns={'age_group_years_start': 'age_start', 'age_group_years_end': 'age_end'})
assert set(age_group_ids) <= set(age_groups.index)
age_groups = age_groups.loc[age_group_ids][["age_start", "age_end"]]
age_groups["age_end"] = age_groups.age_end.replace(125, 100)
age_groups

,age_start,age_end
age_group_id,,
2,0.000000,0.019178
3,0.019178,0.076712
388,0.076712,0.500000
389,0.500000,1.000000
6,5.000000,10.000000
7,10.000000,15.000000
8,15.000000,20.000000
9,20.000000,25.000000
10,25.000000,30.000000


In [15]:
def map_to_gbd_age_groups(df):
    return (
        age_groups.merge(df, how="cross", suffixes=("", "_orig"))
            .pipe(lambda df: df[(df.age_end <= df.age_end_orig) & (df.age_start >= df.age_start_orig)])
            .drop(columns=["age_start_orig", "age_end_orig"])
    )

In [16]:
effective_baseline_coverage.set_index(["sex", "age_start", "age_end", "wealth_quintile", "vehicle", "location"]).index.is_unique

True

In [17]:
effective_counterfactual_coverage.set_index(["sex", "age_start", "age_end", "wealth_quintile", "vehicle", "location"]).index.is_unique

True

In [18]:
effective_baseline_coverage = map_to_gbd_age_groups(effective_baseline_coverage).set_index(index_cols).value
effective_counterfactual_coverage = map_to_gbd_age_groups(effective_counterfactual_coverage).set_index(index_cols).value

In [19]:
delta_effective_coverage = effective_counterfactual_coverage.sub(effective_baseline_coverage)
delta_effective_coverage

location  sex     age_start  age_end     vehicle   wealth_quintile
India     Female  0.0        0.019178    rice      fourth             0.225832
                                                   highest            0.277433
                                                   lowest             0.185656
                                                   middle             0.206280
                                                   second             0.190895
                                                                        ...   
Nigeria   Male    95.0       100.000000  bouillon  fourth             0.097363
                                                   highest            0.069777
                                                   lowest             0.202700
                                                   middle             0.117258
                                                   second             0.159021
Name: value, Length: 460, dtype: float64

In [20]:
def reformat_gbd_data(df):
    index_cols = []
    if "location_id" in df.columns:
        df = df.merge(location_crosswalk, on="location_id").drop(columns=["location_id"])
        index_cols += ["location"]
    if "sex_id" in df.columns:
        df = df.merge(sex_crosswalk, on="sex_id").drop(columns=["sex_id"])
        index_cols += ["sex"]
    if "age_group_id" in df.columns:
        df = df.merge(age_groups, on="age_group_id").drop(columns=["age_group_id"])
        index_cols += ["age_start", "age_end"]
    if len(index_cols) > 0:
        df = df.set_index(index_cols)
    return df

In [21]:
hgb_mean = reformat_gbd_data(
    get_draws('modelable_entity_id',
                    10487,
                    source='epi',
                    location_id=list(location_crosswalk.location_id),
                    age_group_id=age_group_ids,
                    sex_id=list(sex_crosswalk.sex_id),
                    year_id=2021,
                    release_id=9)
)
hgb_mean = hgb_mean[DRAWS].copy()
hgb_mean

,,,,draw_0,draw_1,draw_2,draw_3,draw_4,draw_5,draw_6,draw_7,draw_8,draw_9,...,draw_490,draw_491,draw_492,draw_493,draw_494,draw_495,draw_496,draw_497,draw_498,draw_499
location,sex,age_start,age_end,,,,,,,,,,,,,,,,,,,,,
India,Female,0.000000,0.019178,141.027303,141.174892,143.047093,142.497768,141.204487,141.832669,140.942772,142.891759,141.388006,143.166841,...,140.459189,140.765055,142.503501,140.558958,141.696270,140.565670,141.885428,142.433822,140.370106,142.464730
Nigeria,Female,0.000000,0.019178,140.203331,139.700680,137.486100,143.738463,148.622155,142.898403,135.879741,146.860804,148.441575,136.598751,...,140.399313,143.182558,134.555407,144.192173,135.913431,140.135379,136.145279,139.567376,146.074584,142.051915
India,Male,0.000000,0.019178,146.600654,149.103908,146.074897,145.748027,149.507916,145.964431,146.225025,145.316586,149.278837,147.447709,...,148.331245,145.371190,150.039668,148.060737,146.862862,146.927253,148.502610,148.079533,148.912936,146.613964
Nigeria,Male,0.000000,0.019178,139.655000,152.652888,141.405358,135.236759,145.950384,148.051338,145.753920,143.368754,148.505532,149.271473,...,151.005295,142.522210,132.928140,140.653593,156.208075,152.033406,131.602486,153.680412,146.369116,149.265222
India,Female,0.019178,0.076712,128.185915,129.096397,129.130806,127.546092,129.886294,129.778156,127.898826,129.079936,129.428546,128.289116,...,128.716439,129.137330,128.530514,127.198688,128.109218,127.745823,128.225212,127.911793,129.511153,128.852131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Nigeria,Male,0.076712,0.500000,94.267052,98.335982,96.967303,98.720429,100.392380,97.388312,99.835516,105.483599,97.071404,97.055824,...,96.493032,97.136619,103.343167,97.834196,95.880294,99.447538,99.703969,102.837446,97.052305,103.362419
India,Female,0.500000,1.000000,101.917809,99.823676,100.483745,100.428266,100.642931,100.292157,99.313897,100.007150,99.623193,99.082577,...,101.271734,100.142204,101.161263,101.477375,100.662558,100.274449,100.699653,99.977255,100.885253,99.743914
Nigeria,Female,0.500000,1.000000,100.148802,99.844921,99.645127,100.145850,100.608923,101.012091,100.358797,98.995013,99.743129,100.867444,...,98.288392,99.606637,100.119695,99.217875,98.652748,99.338615,99.382140,98.823520,99.294049,98.803754


In [22]:
hemoglobin_mean_disparities = pd.concat([
    pd.read_csv(f'../0100_data_prep/results/hemoglobin/mean_disparities/{location}.csv')
        .assign(location=location.title())
    for location in locations
])
hemoglobin_mean_disparities = (
    map_to_gbd_age_groups(hemoglobin_mean_disparities[hemoglobin_mean_disparities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["location", "sex", "age_start", "age_end", "wealth_quintile"]).value
)
hemoglobin_mean_disparities

location  sex     age_start  age_end     wealth_quintile
India     Female  0.0        0.019178    lowest             103.655561
                                         second             105.257986
                                         middle             105.397530
                                         fourth             106.765336
                                         highest            107.989017
                                                               ...    
Nigeria   Male    95.0       100.000000  lowest             113.444966
                                         second             115.420130
                                         middle             116.087185
                                         fourth             116.578539
                                         highest            118.793246
Name: value, Length: 460, dtype: float64

In [23]:
wealth_quintile_probabilities = pd.concat([
    pd.read_csv(f'../0100_data_prep/results/wealth_quintile_probabilities/{location}.csv')
        .assign(location=location.title())
    for location in locations
])
wealth_quintile_probabilities


,sex,age_start,age_end,pregnant,lowest,second,middle,fourth,highest,location
0,Female,0.0,5.0,not_pregnant,0.251317,0.220571,0.195838,0.184456,0.147818,India
1,Female,5.0,15.0,not_pregnant,0.269038,0.219459,0.193766,0.172569,0.145167,India
2,Female,15.0,30.0,not_pregnant,0.179314,0.203535,0.211692,0.209712,0.195747,India
3,Female,15.0,30.0,pregnant,0.221262,0.222487,0.214771,0.182324,0.159154,India
4,Female,30.0,50.0,not_pregnant,0.174181,0.188673,0.199713,0.213463,0.223970,India
5,Female,30.0,50.0,pregnant,0.314491,0.182529,0.131633,0.167147,0.204200,India
6,Female,50.0,100.0,not_pregnant,0.188708,0.187540,0.191253,0.199636,0.232863,India
7,Male,0.0,5.0,not_pregnant,0.242600,0.215610,0.200229,0.183923,0.157639,India
8,Male,5.0,15.0,not_pregnant,0.261604,0.217675,0.190718,0.176545,0.153458,India
9,Male,15.0,30.0,not_pregnant,0.169172,0.205794,0.214029,0.208330,0.202675,India


In [24]:
map_to_gbd_age_groups(wealth_quintile_probabilities[wealth_quintile_probabilities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))

,age_start,age_end,sex,lowest,second,middle,fourth,highest,location
0,0.000000,0.019178,Female,0.251317,0.220571,0.195838,0.184456,0.147818,India
5,0.000000,0.019178,Male,0.242600,0.215610,0.200229,0.183923,0.157639,India
10,0.000000,0.019178,Female,0.220707,0.219801,0.207518,0.182416,0.169557,Nigeria
15,0.000000,0.019178,Male,0.214963,0.223211,0.203137,0.188194,0.170495,Nigeria
20,0.019178,0.076712,Female,0.251317,0.220571,0.195838,0.184456,0.147818,India
...,...,...,...,...,...,...,...,...,...
439,90.000000,95.000000,Male,0.209864,0.183658,0.188784,0.202148,0.215546,Nigeria
444,95.000000,100.000000,Female,0.188708,0.187540,0.191253,0.199636,0.232863,India
449,95.000000,100.000000,Male,0.176575,0.184392,0.190461,0.200876,0.247696,India
454,95.000000,100.000000,Female,0.196077,0.184902,0.228885,0.192974,0.197161,Nigeria


In [25]:
wealth_quintile_probabilities = (
    map_to_gbd_age_groups(wealth_quintile_probabilities[wealth_quintile_probabilities.pregnant == "not_pregnant"].drop(columns=["pregnant"]))
        .set_index(["location", "sex", "age_start", "age_end"])
)
wealth_quintile_probabilities.columns.name = 'wealth_quintile'
wealth_quintile_probabilities = wealth_quintile_probabilities.stack()
wealth_quintile_probabilities

location  sex     age_start  age_end     wealth_quintile
India     Female  0.0        0.019178    lowest             0.251317
                                         second             0.220571
                                         middle             0.195838
                                         fourth             0.184456
                                         highest            0.147818
                                                              ...   
Nigeria   Male    95.0       100.000000  lowest             0.209864
                                         second             0.183658
                                         middle             0.188784
                                         fourth             0.202148
                                         highest            0.215546
Length: 460, dtype: float64

In [26]:
pre_disparity_groups = hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum()
pre_disparity_groups

draw_0      draw_1      draw_2  \
location sex    age_start age_end                                          
India    Female 0.000000  0.019178    141.027303  141.174892  143.047093   
                0.019178  0.076712    128.185915  129.096397  129.130806   
                0.076712  0.500000    103.541148  104.218523  102.321094   
                0.500000  1.000000    101.917809   99.823676  100.483745   
                5.000000  10.000000   112.932920  111.768831  112.598540   
...                                          ...         ...         ...   
Nigeria  Male   75.000000 80.000000   125.293565  127.635808  121.016674   
                80.000000 85.000000   136.438006  140.853010  131.367813   
                85.000000 90.000000   133.029351  120.328832  136.611180   
                90.000000 95.000000   116.866389  104.222078  111.737153   
                95.000000 100.000000  104.414752  104.110723   99.482286   

                                          draw_3      draw_4      draw_5  \
location sex    age_start age_end                                          
India    Female 0.000000  0.019178    142.497768  141.204487  141.832669   
                0.019178  0.076712    127.546092  129.886294  129.778156   
                0.076712  0.500000    104.327579  103.623847  103.637978   
                0.500000  1.000000    100.428266  100.642931  100.292157   
                5.000000  10.000000   114.147894  113.027698  112.048307   
...                                          ...         ...         ...   
Nigeria  Male   75.000000 80.000000   127.115747  138.057306  124.135326   
                80.000000 85.000000   139.943361  136.350483  131.038066   
                85.000000 90.000000   126.391697  129.370208  136.797710   
                90.000000 95.000000   113.207507  106.352527   98.297926   
                95.000000 100.000000  102.887575  107.603354  105.459291   

                                          draw_6      draw_7      draw_8  \
location sex    age_start age_end                                          
India    Female 0.000000  0.019178    140.942772  142.891759  141.388006   
                0.019178  0.076712    127.898826  129.079936  129.428546   
                0.076712  0.500000    103.094638  103.816330  101.953160   
                0.500000  1.000000     99.313897  100.007150   99.623193   
                5.000000  10.000000   112.435412  112.251242  114.540273   
...                                          ...         ...         ...   
Nigeria  Male   75.000000 80.000000   131.351376  140.422146  134.386319   
                80.000000 85.000000   123.096959  137.390001  135.796309   
                85.000000 90.000000   127.729441  121.481892  124.340741   
                90.000000 95.000000   113.236702  105.723986  114.701460   
                95.000000 100.000000  103.255580  110.455551  105.846010   

                                          draw_9  ...    draw_490    draw_491  \
location sex    age_start age_end                 ...                           
India    Female 0.000000  0.019178    143.166841  ...  140.459189  140.765055   
                0.019178  0.076712    128.289116  ...  128.716439  129.137330   
                0.076712  0.500000    104.552078  ...  102.776895  103.279928   
                0.500000  1.000000     99.082577  ...  101.271734  100.142204   
                5.000000  10.000000   112.764552  ...  113.947008  111.993490   
...                                          ...  ...         ...         ...   
Nigeria  Male   75.000000 80.000000   133.184690  ...  136.984105  141.082724   
                80.000000 85.000000   130.850945  ...  126.774249  139.078352   
                85.000000 90.000000   129.907846  ...  123.342804  132.476853   
                90.000000 95.000000   109.299417  ...  106.352476  105.521819   
                95.000000 100.000000  113.724775  ...  106.311908  106.971579   

                     

In [27]:
hgb_mean = hgb_mean.mul(hemoglobin_mean_disparities, axis=0)

In [28]:
scale_factor = pre_disparity_groups / hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum()
scale_factor

draw_0    draw_1    draw_2    draw_3  \
location sex    age_start age_end                                              
India    Female 0.000000  0.019178    0.009473  0.009473  0.009473  0.009473   
                0.019178  0.076712    0.009473  0.009473  0.009473  0.009473   
                0.076712  0.500000    0.009473  0.009473  0.009473  0.009473   
                0.500000  1.000000    0.009473  0.009473  0.009473  0.009473   
                5.000000  10.000000   0.009011  0.009011  0.009011  0.009011   
...                                        ...       ...       ...       ...   
Nigeria  Male   75.000000 80.000000   0.008614  0.008614  0.008614  0.008614   
                80.000000 85.000000   0.008614  0.008614  0.008614  0.008614   
                85.000000 90.000000   0.008614  0.008614  0.008614  0.008614   
                90.000000 95.000000   0.008614  0.008614  0.008614  0.008614   
                95.000000 100.000000  0.008614  0.008614  0.008614  0.008614   

                                        draw_4    draw_5    draw_6    draw_7  \
location sex    age_start age_end                                              
India    Female 0.000000  0.019178    0.009473  0.009473  0.009473  0.009473   
                0.019178  0.076712    0.009473  0.009473  0.009473  0.009473   
                0.076712  0.500000    0.009473  0.009473  0.009473  0.009473   
                0.500000  1.000000    0.009473  0.009473  0.009473  0.009473   
                5.000000  10.000000   0.009011  0.009011  0.009011  0.009011   
...                                        ...       ...       ...       ...   
Nigeria  Male   75.000000 80.000000   0.008614  0.008614  0.008614  0.008614   
                80.000000 85.000000   0.008614  0.008614  0.008614  0.008614   
                85.000000 90.000000   0.008614  0.008614  0.008614  0.008614   
                90.000000 95.000000   0.008614  0.008614  0.008614  0.008614   
                95.000000 100.000000  0.008614  0.008614  0.008614  0.008614   

                                        draw_8    draw_9  ...  draw_490  \
location sex    age_start age_end                         ...             
India    Female 0.000000  0.019178    0.009473  0.009473  ...  0.009473   
                0.019178  0.076712    0.009473  0.009473  ...  0.009473   
                0.076712  0.500000    0.009473  0.009473  ...  0.009473   
                0.500000  1.000000    0.009473  0.009473  ...  0.009473   
                5.000000  10.000000   0.009011  0.009011  ...  0.009011   
...                                        ...       ...  ...       ...   
Nigeria  Male   75.000000 80.000000   0.008614  0.008614  ...  0.008614   
                80.000000 85.000000   0.008614  0.008614  ...  0.008614   
                85.000000 90.000000   0.008614  0.008614  ...  0.008614   
                90.000000 95.000000   0.008614  0.008614  ...  0.008614   
                95.000000 100.000000  0.008614  0.008614  ...  0.008614   

                                      draw_491  draw_492  draw_493  draw_494  \
location sex    age_start age_end                                              
India    Female 0.000000  0.019178    0.009473  0.009473  0.009473  0.009473   
                0.019178  0.076712    0.009473  0.009473  0.009473  0.009473   
                0.076712  0.500000    0.009473  0.009473  0.009473  0.009473   
                0.500000  1.000000    0.009473  0.009473  0.009473  0.009473   
                5.000000  10.000000   0.009011  0.009011  0.009011  0.009011   
...                                        ...       ...       ...       ...   
Nigeria  Male   75.000000 80.000000   0.008614  0.008614  0.008614  0.008614   
                80.000000 85.000000   0.008614  0.008614  0.008614  0.008614   
                85.000000 90.000000   0.008614  0.008614  0.008614  0.008614   
                90.000000 95.000000   0.008614  0.008614  0.008614  0.008614   
                95.000

In [29]:
hgb_mean = hgb_mean * scale_factor

In [30]:
assert np.allclose(
    hgb_mean.mul(wealth_quintile_probabilities, axis=0).groupby([c for c in hgb_mean.index.names if c != 'wealth_quintile']).sum(),
    pre_disparity_groups,
)

In [31]:
hgb_mean = (
    hgb_mean.reset_index()
        .assign(vehicle=lambda x: x.location.map({"India": 'rice', "Nigeria": 'bouillon'}))
        .set_index(list(hgb_mean.index.names) + ['vehicle'])
)
hgb_mean

draw_0  \
location sex    age_start age_end    wealth_quintile vehicle                
India    Female 0.0       0.019178   lowest          rice      138.477305   
                                     second          rice      140.618044   
                                     middle          rice      140.804466   
                                     fourth          rice      142.631768   
                                     highest         rice      144.266529   
...                                                                   ...   
Nigeria  Male   95.0      100.000000 lowest          bouillon  102.033291   
                                     second          bouillon  103.809769   
                                     middle          bouillon  104.409723   
                                     fourth          bouillon  104.851652   
                                     highest         bouillon  106.843576   

                                                                   draw_1  \
location sex    age_start age_end    wealth_quintile vehicle                
India    Female 0.0       0.019178   lowest          rice      138.622225   
                                     second          rice      140.765205   
                                     middle          rice      140.951821   
                                     fourth          rice      142.781036   
                                     highest         rice      144.417508   
...                                                                   ...   
Nigeria  Male   95.0      100.000000 lowest          bouillon  101.736197   
                                     second          bouillon  103.507502   
                                     middle          bouillon  104.105710   
                                     fourth          bouillon  104.546351   
                                     highest         bouillon  106.532475   

                                                                   draw_2  \
location sex    age_start age_end    wealth_quintile vehicle                
India    Female 0.0       0.019178   lowest          rice      140.460574   
                                     second          rice      142.631972   
                                     middle          rice      142.821064   
                                     fourth          rice      144.674537   
                                     highest         rice      146.332711   
...                                                                   ...   
Nigeria  Male   95.0      100.000000 lowest          bouillon   97.213324   
                                     second          bouillon   98.905882   
                                     middle          bouillon   99.477495   
                                     fourth          bouillon   99.898547   
                                     highest         bouillon  101.796374   

                                                                   draw_3  \
location sex    age_start age_end    wealth_quintile vehicle                
India    Female 0.0       0.019178   lowest          rice      139.921181   
                                     second          rice      142.084241   
                                     middle          rice      142.272607   
                                     fourth          rice      144.118962   
                                     highest         rice      145.770769   
...                                                                   ...   
Nigeria  Male   95.0      100.000000 lowest          bouillon  100.540946   
                                     second          bouillon  102.291441   
                                     middle          bouillon  102.882620   
                                     fourth          bouillon  103.318085   
                                     highest         bouillon  105.280875   

                                                           

In [32]:
counterfactual_hgb_mean = hgb_mean.add(delta_effective_coverage * mean_difference, axis=0)
counterfactual_hgb_mean

draw_0  \
location sex    age_start age_end    wealth_quintile vehicle                
India    Female 0.0       0.019178   fourth          rice      143.365723   
                                     highest         rice      145.168187   
                                     lowest          rice      139.080689   
                                     middle          rice      141.474875   
                                     second          rice      141.238453   
...                                                                   ...   
Nigeria  Male   95.0      100.000000 fourth          bouillon  105.260577   
                                     highest         bouillon  107.136639   
                                     lowest          bouillon  102.884631   
                                     middle          bouillon  104.902207   
                                     second          bouillon  104.477659   

                                                                   draw_1  \
location sex    age_start age_end    wealth_quintile vehicle                
India    Female 0.0       0.019178   fourth          rice      143.514991   
                                     highest         rice      145.319166   
                                     lowest          rice      139.225609   
                                     middle          rice      141.622231   
                                     second          rice      141.385614   
...                                                                   ...   
Nigeria  Male   95.0      100.000000 fourth          bouillon  104.955276   
                                     highest         bouillon  106.825538   
                                     lowest          bouillon  102.587537   
                                     middle          bouillon  104.598194   
                                     second          bouillon  104.175392   

                                                                   draw_2  \
location sex    age_start age_end    wealth_quintile vehicle                
India    Female 0.0       0.019178   fourth          rice      145.408492   
                                     highest         rice      147.234369   
                                     lowest          rice      141.063957   
                                     middle          rice      143.491473   
                                     second          rice      143.252381   
...                                                                   ...   
Nigeria  Male   95.0      100.000000 fourth          bouillon  100.307472   
                                     highest         bouillon  102.089437   
                                     lowest          bouillon   98.064663   
                                     middle          bouillon   99.969979   
                                     second          bouillon   99.573772   

                                                                   draw_3  \
location sex    age_start age_end    wealth_quintile vehicle                
India    Female 0.0       0.019178   fourth          rice      144.852917   
                                     highest         rice      146.672426   
                                     lowest          rice      140.524565   
                                     middle          rice      142.943016   
                                     second          rice      142.704650   
...                                                                   ...   
Nigeria  Male   95.0      100.000000 fourth          bouillon  103.727010   
                                     highest         bouillon  105.573938   
                                     lowest          bouillon  101.392286   
                                     middle          bouillon  103.375104   
                                     second          bouillon  102.959331   

                                                           

In [33]:
hgb_mean.columns.name = "draw"
hgb_mean = hgb_mean.stack().rename("mean").reset_index()

In [34]:
counterfactual_hgb_mean.columns.name = "draw"
counterfactual_hgb_mean = counterfactual_hgb_mean.stack().rename("mean").reset_index()

In [35]:
hgb_sd = reformat_gbd_data(
    get_draws('modelable_entity_id',
                10488,
                source='epi',
                location_id=list(location_crosswalk.location_id),
                age_group_id=age_group_ids,
                sex_id=list(sex_crosswalk.sex_id),
                year_id=2021,
                release_id=9)
)

hgb_sd = hgb_sd[DRAWS].copy()
hgb_sd

,,,,draw_0,draw_1,draw_2,draw_3,draw_4,draw_5,draw_6,draw_7,draw_8,draw_9,...,draw_490,draw_491,draw_492,draw_493,draw_494,draw_495,draw_496,draw_497,draw_498,draw_499
location,sex,age_start,age_end,,,,,,,,,,,,,,,,,,,,,
India,Female,0.000000,0.019178,11.056572,15.419496,12.937057,14.108613,13.020095,12.577809,17.754828,13.099648,11.912721,13.534418,...,10.329697,10.668878,13.046126,10.782160,11.950843,10.353358,20.932112,15.441125,15.067467,12.300862
Nigeria,Female,0.000000,0.019178,26.026960,23.907182,16.403169,18.715768,17.036223,21.455689,13.328800,27.165364,32.583905,8.017943,...,25.491270,12.187820,14.047787,26.560128,18.689541,12.964533,19.262095,9.520727,18.461955,21.695176
India,Male,0.000000,0.019178,7.610899,6.575979,7.453095,6.250333,7.920491,6.340217,6.466164,6.139428,5.691987,7.091346,...,7.533375,6.431263,7.397510,7.293675,7.932027,7.802585,6.921468,7.639538,8.448905,8.150727
Nigeria,Male,0.000000,0.019178,8.741365,8.444370,10.058485,9.269974,11.570569,8.714296,6.731408,7.346581,8.429818,6.594639,...,6.056589,10.017631,12.212247,11.265512,11.507346,8.898354,16.829171,19.359118,7.850751,10.302650
India,Female,0.019178,0.076712,18.143206,17.170122,18.638921,21.675732,17.362276,18.653702,20.182212,17.627105,18.478847,19.942071,...,18.396669,19.537466,16.708525,20.642018,22.565014,22.806860,18.916546,19.554642,17.105346,19.331287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Nigeria,Male,0.076712,0.500000,10.834561,14.464839,16.716692,13.781233,14.308120,18.266908,13.962939,16.882978,15.100336,13.834311,...,11.962733,16.352547,21.153987,12.123287,16.187477,11.642224,18.578292,17.016439,19.081475,19.191454
India,Female,0.500000,1.000000,13.756756,14.711771,13.648304,13.187534,13.376081,14.524967,13.737699,14.473522,14.511934,14.818950,...,13.570370,14.580505,13.474202,14.509005,13.042663,14.323584,13.957877,14.127909,13.305026,13.191365
Nigeria,Female,0.500000,1.000000,15.088749,16.763458,14.467460,14.968448,14.433290,15.700162,16.891376,14.383653,13.580970,13.668688,...,13.607053,14.132132,15.332481,15.166399,15.459204,13.412401,14.305163,15.815794,15.249249,13.555534


In [36]:
hgb_sd.columns.name = "draw"
hgb_sd = hgb_sd.stack().rename("sd").reset_index()
hgb_sd

,location,sex,age_start,age_end,draw,sd
0,India,Female,0.0,0.019178,draw_0,11.056572
1,India,Female,0.0,0.019178,draw_1,15.419496
2,India,Female,0.0,0.019178,draw_2,12.937057
3,India,Female,0.0,0.019178,draw_3,14.108613
4,India,Female,0.0,0.019178,draw_4,13.020095
...,...,...,...,...,...,...
45995,Nigeria,Male,0.5,1.000000,draw_495,15.843389
45996,Nigeria,Male,0.5,1.000000,draw_496,17.233076
45997,Nigeria,Male,0.5,1.000000,draw_497,13.225374
45998,Nigeria,Male,0.5,1.000000,draw_498,17.182869


In [37]:
import risk_distributions

In [38]:
hgb_sd

,location,sex,age_start,age_end,draw,sd
0,India,Female,0.0,0.019178,draw_0,11.056572
1,India,Female,0.0,0.019178,draw_1,15.419496
2,India,Female,0.0,0.019178,draw_2,12.937057
3,India,Female,0.0,0.019178,draw_3,14.108613
4,India,Female,0.0,0.019178,draw_4,13.020095
...,...,...,...,...,...,...
45995,Nigeria,Male,0.5,1.000000,draw_495,15.843389
45996,Nigeria,Male,0.5,1.000000,draw_496,17.233076
45997,Nigeria,Male,0.5,1.000000,draw_497,13.225374
45998,Nigeria,Male,0.5,1.000000,draw_498,17.182869


In [39]:
hgb_mean

,location,sex,age_start,age_end,wealth_quintile,vehicle,draw,mean
0,India,Female,0.0,0.019178,lowest,rice,draw_0,138.477305
1,India,Female,0.0,0.019178,lowest,rice,draw_1,138.622225
2,India,Female,0.0,0.019178,lowest,rice,draw_2,140.460574
3,India,Female,0.0,0.019178,lowest,rice,draw_3,139.921181
4,India,Female,0.0,0.019178,lowest,rice,draw_4,138.651285
...,...,...,...,...,...,...,...,...
229995,Nigeria,Male,95.0,100.000000,highest,bouillon,draw_495,99.179462
229996,Nigeria,Male,95.0,100.000000,highest,bouillon,draw_496,108.553528
229997,Nigeria,Male,95.0,100.000000,highest,bouillon,draw_497,107.702608
229998,Nigeria,Male,95.0,100.000000,highest,bouillon,draw_498,101.506707


In [40]:
mean_and_sd_hgb = pd.concat([
    hgb_mean.merge(hgb_sd, how="outer", validate="m:1").assign(scenario='baseline'),
    counterfactual_hgb_mean.merge(hgb_sd, how="outer", validate="m:1").assign(scenario='intervention')
])
mean_and_sd_hgb

,location,sex,age_start,age_end,wealth_quintile,vehicle,draw,mean,sd,scenario
0,India,Female,0.0,0.019178,lowest,rice,draw_0,138.477305,11.056572,baseline
1,India,Female,0.0,0.019178,second,rice,draw_0,140.618044,11.056572,baseline
2,India,Female,0.0,0.019178,middle,rice,draw_0,140.804466,11.056572,baseline
3,India,Female,0.0,0.019178,fourth,rice,draw_0,142.631768,11.056572,baseline
4,India,Female,0.0,0.019178,highest,rice,draw_0,144.266529,11.056572,baseline
...,...,...,...,...,...,...,...,...,...,...
229995,Nigeria,Male,95.0,100.000000,fourth,bouillon,draw_499,108.137248,17.423849,intervention
229996,Nigeria,Male,95.0,100.000000,highest,bouillon,draw_499,110.067959,17.423849,intervention
229997,Nigeria,Male,95.0,100.000000,lowest,bouillon,draw_499,105.683979,17.423849,intervention
229998,Nigeria,Male,95.0,100.000000,middle,bouillon,draw_499,107.766754,17.423849,intervention


In [41]:
thresholds = reformat_gbd_data(
    pd.read_csv('/share/mnch/anemia/code/reference/model/anemia_thresholds.csv')
)
thresholds

,,,age_group_name,pregnant,grp,hgb_lower_mild,hgb_upper_mild,hgb_lower_moderate,hgb_upper_moderate,hgb_lower_severe,hgb_upper_severe,hgb_lower_anemic,hgb_upper_anemic
sex,age_start,age_end,,,,,,,,,,,
Female,0.000000,0.019178,Early Neonatal,0,under_5,145,160,100,145,0,100,0,160
Male,0.000000,0.019178,Early Neonatal,0,under_5,145,160,100,145,0,100,0,160
Female,0.019178,0.076712,Late Neonatal,0,under_5,120,135,85,120,0,85,0,135
Male,0.019178,0.076712,Late Neonatal,0,under_5,120,135,85,120,0,85,0,135
Female,0.076712,0.500000,1-5 months,0,under_5,100,110,70,100,0,70,0,110
Male,0.076712,0.500000,1-5 months,0,under_5,100,110,70,100,0,70,0,110
Female,0.500000,1.000000,6-11 months,0,under_5,100,110,70,100,0,70,0,110
Male,0.500000,1.000000,6-11 months,0,under_5,100,110,70,100,0,70,0,110
Female,5.000000,10.000000,5 to 9,0,sa_child,110,115,80,110,0,80,0,115


In [42]:
mean_and_sd_hgb = mean_and_sd_hgb.assign(pregnant=0).merge(thresholds, on=["sex", "age_start", "age_end", "pregnant"], how="left", validate="m:1")
mean_and_sd_hgb

,location,sex,age_start,age_end,wealth_quintile,vehicle,draw,mean,sd,scenario,...,age_group_name,grp,hgb_lower_mild,hgb_upper_mild,hgb_lower_moderate,hgb_upper_moderate,hgb_lower_severe,hgb_upper_severe,hgb_lower_anemic,hgb_upper_anemic
0,India,Female,0.0,0.019178,lowest,rice,draw_0,138.477305,11.056572,baseline,...,Early Neonatal,under_5,145,160,100,145,0,100,0,160
1,India,Female,0.0,0.019178,second,rice,draw_0,140.618044,11.056572,baseline,...,Early Neonatal,under_5,145,160,100,145,0,100,0,160
2,India,Female,0.0,0.019178,middle,rice,draw_0,140.804466,11.056572,baseline,...,Early Neonatal,under_5,145,160,100,145,0,100,0,160
3,India,Female,0.0,0.019178,fourth,rice,draw_0,142.631768,11.056572,baseline,...,Early Neonatal,under_5,145,160,100,145,0,100,0,160
4,India,Female,0.0,0.019178,highest,rice,draw_0,144.266529,11.056572,baseline,...,Early Neonatal,under_5,145,160,100,145,0,100,0,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459995,Nigeria,Male,95.0,100.000000,fourth,bouillon,draw_499,108.137248,17.423849,intervention,...,95 plus,adult_male,110,130,80,110,0,80,0,130
459996,Nigeria,Male,95.0,100.000000,highest,bouillon,draw_499,110.067959,17.423849,intervention,...,95 plus,adult_male,110,130,80,110,0,80,0,130
459997,Nigeria,Male,95.0,100.000000,lowest,bouillon,draw_499,105.683979,17.423849,intervention,...,95 plus,adult_male,110,130,80,110,0,80,0,130
459998,Nigeria,Male,95.0,100.000000,middle,bouillon,draw_499,107.766754,17.423849,intervention,...,95 plus,adult_male,110,130,80,110,0,80,0,130


In [43]:
assert (
    (mean_and_sd_hgb.hgb_upper_mild == mean_and_sd_hgb.hgb_upper_anemic).all() &
    (mean_and_sd_hgb.hgb_lower_severe == mean_and_sd_hgb.hgb_lower_anemic).all()
)
mean_and_sd_hgb = mean_and_sd_hgb.drop(columns=["hgb_upper_anemic", "hgb_lower_anemic"])

In [44]:
assert (
    (mean_and_sd_hgb.hgb_lower_mild == mean_and_sd_hgb.hgb_upper_moderate).all() &
    (mean_and_sd_hgb.hgb_lower_moderate == mean_and_sd_hgb.hgb_upper_severe).all()
)
mean_and_sd_hgb = mean_and_sd_hgb.drop(columns=["hgb_lower_mild", "hgb_lower_moderate"])

In [45]:
def _hemoglobin_distribution_parts_from_mean_sd(mean, sd):
    # NOTE: This is an unusual ensemble distribution. We should add functionality to the
    # EnsembleDistribution class to make this easier.
    x_min = 0
    x_max = 220
    gamma_params = risk_distributions.risk_distributions.Gamma.get_parameters(
        mean=mean, sd=sd
    )
    # NOTE: We have to override these, otherwise Gamma is overly conservative in what values
    # are computable
    # https://github.com/ihmeuw/risk_distributions/issues/61
    gamma_params["x_min"] = x_min
    gamma_params["x_max"] = x_max
    hemoglobin_distribution_gamma_part = risk_distributions.risk_distributions.Gamma(
        gamma_params
    )

    # NOTE: Forced to duplicate https://github.com/ihmeuw/risk_distributions/blob/a9ed9d7e8372590018355012a7a7ffefa87b0819/src/risk_distributions/risk_distributions.py#L428-L434
    # because it doesn't permit the custom x_min and x_max, and these are used in calculating the others
    mgumbel_params = pd.DataFrame({
        "loc": x_max - mean - (np.euler_gamma * np.sqrt(6) / np.pi * sd),
        "scale": np.sqrt(6) / np.pi * sd,
        "x_min": x_min,
        "x_max": x_max,
    })
    hemoglobin_distribution_mgumbel_part = (
        risk_distributions.risk_distributions.MirroredGumbel(mgumbel_params)
    )
    return hemoglobin_distribution_gamma_part, hemoglobin_distribution_mgumbel_part

(
    hemoglobin_distribution_gamma_part,
    hemoglobin_distribution_mgumbel_part,
) = _hemoglobin_distribution_parts_from_mean_sd(mean_and_sd_hgb['mean'], mean_and_sd_hgb.sd)

def cdf(x):
    gamma_cdf = hemoglobin_distribution_gamma_part.cdf(x)
    # NOTE: There is a bug in this CDF function -- it is reversed!
    # https://github.com/ihmeuw/risk_distributions/issues/62
    mgumbel_cdf = 1 - hemoglobin_distribution_mgumbel_part.cdf(x)
    return (
        0.4
        * gamma_cdf
        + 0.6
        * mgumbel_cdf
    )

In [46]:
mean_and_sd_hgb["severe"] = cdf(mean_and_sd_hgb.hgb_upper_severe.copy()) - cdf(mean_and_sd_hgb.hgb_lower_severe.copy())
mean_and_sd_hgb["moderate"] = cdf(mean_and_sd_hgb.hgb_upper_moderate.copy()) - mean_and_sd_hgb["severe"].copy()
mean_and_sd_hgb["mild"] = cdf(mean_and_sd_hgb.hgb_upper_mild.copy()) - mean_and_sd_hgb["moderate"].copy() - mean_and_sd_hgb["severe"].copy()
mean_and_sd_hgb["anemic"] = mean_and_sd_hgb["mild"] + mean_and_sd_hgb["moderate"] + mean_and_sd_hgb["severe"]
mean_and_sd_hgb

,location,sex,age_start,age_end,wealth_quintile,vehicle,draw,mean,sd,scenario,...,age_group_name,grp,hgb_upper_mild,hgb_upper_moderate,hgb_lower_severe,hgb_upper_severe,severe,moderate,mild,anemic
0,India,Female,0.0,0.019178,lowest,rice,draw_0,138.477305,11.056572,baseline,...,Early Neonatal,under_5,160,145,0,100,0.003894,0.706046,0.277390,0.987329
1,India,Female,0.0,0.019178,second,rice,draw_0,140.618044,11.056572,baseline,...,Early Neonatal,under_5,160,145,0,100,0.003030,0.625916,0.350447,0.979392
2,India,Female,0.0,0.019178,middle,rice,draw_0,140.804466,11.056572,baseline,...,Early Neonatal,under_5,160,145,0,100,0.002965,0.618794,0.356695,0.978454
3,India,Female,0.0,0.019178,fourth,rice,draw_0,142.631768,11.056572,baseline,...,Early Neonatal,under_5,160,145,0,100,0.002396,0.548791,0.415040,0.966227
4,India,Female,0.0,0.019178,highest,rice,draw_0,144.266529,11.056572,baseline,...,Early Neonatal,under_5,160,145,0,100,0.001982,0.487061,0.460202,0.949245
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
459995,Nigeria,Male,95.0,100.000000,fourth,bouillon,draw_499,108.137248,17.423849,intervention,...,95 plus,adult_male,130,110,0,80,0.057745,0.452552,0.409866,0.920163
459996,Nigeria,Male,95.0,100.000000,highest,bouillon,draw_499,110.067959,17.423849,intervention,...,95 plus,adult_male,130,110,0,80,0.048209,0.416406,0.431422,0.896036
459997,Nigeria,Male,95.0,100.000000,lowest,bouillon,draw_499,105.683979,17.423849,intervention,...,95 plus,adult_male,130,110,0,80,0.072486,0.497147,0.374489,0.944122
459998,Nigeria,Male,95.0,100.000000,middle,bouillon,draw_499,107.766754,17.423849,intervention,...,95 plus,adult_male,130,110,0,80,0.059773,0.459413,0.405066,0.924252


In [47]:
disability_weights = pd.read_hdf('/mnt/team/simulation_science/costeffectiveness/auxiliary_data/GBD_2021/02_processed_data/disability_weight/sequela/all/all.hdf')
disability_weights = disability_weights[disability_weights.healthstate.isin(['anemia_mild', 'anemia_mod', 'anemia_sev'])].set_index('healthstate').filter(like='draw_')
disability_weights.columns.name = 'draw'
disability_weights = disability_weights.stack().rename('disability_weight').reset_index()
disability_weights

,healthstate,draw,disability_weight
0,anemia_mild,draw_0,0.002420
1,anemia_mild,draw_1,0.003172
2,anemia_mild,draw_2,0.002644
3,anemia_mild,draw_3,0.003085
4,anemia_mild,draw_4,0.001845
...,...,...,...
2995,anemia_sev,draw_995,0.174969
2996,anemia_sev,draw_996,0.086798
2997,anemia_sev,draw_997,0.131996
2998,anemia_sev,draw_998,0.124427


In [48]:
mean_and_sd_hgb = mean_and_sd_hgb.merge(
    disability_weights[disability_weights.healthstate == 'anemia_mild'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'mild_dw'}),
    validate="m:1",
).merge(
    disability_weights[disability_weights.healthstate == 'anemia_mod'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'moderate_dw'}),
    validate="m:1",
).merge(
    disability_weights[disability_weights.healthstate == 'anemia_sev'][['draw', 'disability_weight']].rename(columns={'disability_weight': 'severe_dw'}),
    validate="m:1",
)

In [49]:
mean_and_sd_hgb["mild_ylds"] = mean_and_sd_hgb.mild * mean_and_sd_hgb.mild_dw
mean_and_sd_hgb["moderate_ylds"] = mean_and_sd_hgb.moderate * mean_and_sd_hgb.moderate_dw
mean_and_sd_hgb["severe_ylds"] = mean_and_sd_hgb.severe * mean_and_sd_hgb.severe_dw
mean_and_sd_hgb['anemic_ylds'] = mean_and_sd_hgb['mild_ylds'] + mean_and_sd_hgb['moderate_ylds'] + mean_and_sd_hgb['severe_ylds']

In [50]:
index_cols = ['location', 'age_start', 'age_end', 'sex', 'draw', 'vehicle', "wealth_quintile"]
value_cols = ["mild", "moderate", "severe", "anemic", "mild_ylds", "moderate_ylds", "severe_ylds", "anemic_ylds"]

baseline_anemia = mean_and_sd_hgb[mean_and_sd_hgb.scenario == 'baseline'].set_index(index_cols)[value_cols]
counterfactual_anemia = mean_and_sd_hgb[mean_and_sd_hgb.scenario == 'intervention'].set_index(index_cols)[value_cols]

In [51]:
baseline_anemia.sort_index()

mild  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.415040   
                                                      highest          0.460202   
                                                      lowest           0.277390   
                                                      middle           0.356695   
                                                      second           0.350447   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.308946   
                                                      highest          0.327652   
                                                      lowest           0.277851   
                                                      middle           0.304404   
                                                      second           0.298027   

                                                                       moderate  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.548791   
                                                      highest          0.487061   
                                                      lowest           0.706046   
                                                      middle           0.618794   
                                                      second           0.625916   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.468263   
                                                      highest          0.448116   
                                                      lowest           0.493218   
                                                      middle           0.472472   
                                                      second           0.478019   

                                                                         severe  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.002396   
                                                      highest          0.001982   
                                                      lowest           0.003894   
                                                      middle           0.002965   
                                                      second           0.003030   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.141823   
                                                      highest          0.124332   
                                                      lowest           0.169661   
                                                      middle           0.145947   
                                                      second           0.151688   

                                                                         anemic  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.966227   
                                                      highest          0.949245   
                                                      lowest           0.987329   
                                                      middle           0.978454   
                                                      second           0.979392   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.919033   
                                                      highest          0.900100   
     

In [52]:
counterfactual_anemia.sort_index()

mild  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.436339   
                                                      highest          0.481151   
                                                      lowest           0.298026   
                                                      middle           0.378799   
                                                      second           0.371078   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.313116   
                                                      highest          0.330200   
                                                      lowest           0.287961   
                                                      middle           0.309567   
                                                      second           0.305264   

                                                                       moderate  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.520881   
                                                      highest          0.453881   
                                                      lowest           0.683881   
                                                      middle           0.593111   
                                                      second           0.602177   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.464191   
                                                      highest          0.444935   
                                                      lowest           0.486012   
                                                      middle           0.467670   
                                                      second           0.471692   

                                                                         severe  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.002200   
                                                      highest          0.001785   
                                                      lowest           0.003628   
                                                      middle           0.002742   
                                                      second           0.002818   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.138005   
                                                      highest          0.121854   
                                                      lowest           0.160675   
                                                      middle           0.141257   
                                                      second           0.145168   

                                                                         anemic  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.959420   
                                                      highest          0.936817   
                                                      lowest           0.985535   
                                                      middle           0.974651   
                                                      second           0.976073   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.915312   
                                                      highest          0.896989   
     

In [53]:
baseline_anemia - counterfactual_anemia

mild  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth          -0.021299   
                                                      highest         -0.020949   
                                                      lowest          -0.020637   
                                                      middle          -0.022103   
                                                      second          -0.020631   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth          -0.004170   
                                                      highest         -0.002548   
                                                      lowest          -0.010110   
                                                      middle          -0.005163   
                                                      second          -0.007237   

                                                                       moderate  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.027910   
                                                      highest          0.033180   
                                                      lowest           0.022165   
                                                      middle           0.025683   
                                                      second           0.023739   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.004072   
                                                      highest          0.003181   
                                                      lowest           0.007207   
                                                      middle           0.004802   
                                                      second           0.006326   

                                                                         severe  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.000196   
                                                      highest          0.000197   
                                                      lowest           0.000267   
                                                      middle           0.000223   
                                                      second           0.000212   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.003818   
                                                      highest          0.002477   
                                                      lowest           0.008986   
                                                      middle           0.004690   
                                                      second           0.006520   

                                                                         anemic  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.006808   
                                                      highest          0.012428   
                                                      lowest           0.001794   
                                                      middle           0.003803   
                                                      second           0.003319   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.003721   
                                                      highest          0.003111   
     

In [54]:
# BUT our counterfactual is only in a world where everyone is iron-responsive.
iron_responsive_anemia_sequelae = [1004, 1005, 1006, 1008, 1009, 1010, 1012, 1013, 
                                   1014, 1016, 1017, 1018, 1020, 1021, 1022, 1024, 1025, 1026, 
                                   1028, 1029, 1030, 1032, 1033, 1034, 1361, 1364, 1367, 1373, 1376, 
                                   1379, 1385, 1388, 1391, 1397, 1400, 1403, 1409, 1412, 1415, 1421, 
                                   1424, 1427, 1433, 1436, 1439, 1445, 1448, 1451, 5213, 5216, 5219, 
                                   5222, 5225, 5228, 5237, 5240, 5243, 5246, 5249, 5252, 5261, 5264, 
                                   5267, 5270, 5273, 5276, 4985, 4988, 4991, 4994, 4997, 5000, 5009, 
                                   5012, 5015, 5678, 5681, 5684, 7214, 7217, 7220, 4952, 4955, 4958, 
                                   4961, 4964, 4967, 4976, 4979, 4982, 5627, 5630, 5633, 7202, 7205, 
                                   7208, 5393, 5396, 5399, 182, 183, 184, 240, 241, 242, 177, 178, 
                                   179, 144,145,146,172,173,174,525,526,527,1106,1107,1108,537,538,
                                   539,206,207,208, 22989, 22990, 22991, 22992, 22993, 22999, 23000, 
                                   23001, 23002, 23003, 23009, 23010, 23011, 23012, 23013,
                                   5567, 5570, 5573, 5579, 5582, 5585,
                                   23030, 23031, 23032, 23034, 23035, 23036, 23038, 23039, 23040,
                                   23042, 23043, 23044, 23046, 23047, 23048]

ira_prev = reformat_gbd_data(get_draws('sequela_id', iron_responsive_anemia_sequelae, 
                 source='como',
                 location_id=list(location_crosswalk.location_id),
                 age_group_id=age_group_ids,
                 sex_id=list(sex_crosswalk.sex_id),
                 year_id=2021,
                 release_id=9))
ira_prev = ira_prev.groupby(['location', 'sex', 'age_start', 'age_end']).sum().filter(like="draw")
ira_prev

/ihme/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/5000_analyze_results/.venv/lib/python3.11/site-packages/get_draws/api.py:192: UserWarning: No version_id was specified, so get_draws will automatically determine a best version ID to use from the given parameters. If you want to retrieve a specific version please pass a version_id directly.
  warnings.warn(


draw_0    draw_1   draw_10  draw_100  \
location sex    age_start age_end                                              
India    Female 0.000000  0.019178    0.899755  0.857321  0.874569  0.887292   
                0.019178  0.076712    0.554685  0.561048  0.597522  0.581726   
                0.076712  0.500000    0.643967  0.624061  0.637449  0.645793   
                0.500000  1.000000    0.725464  0.750456  0.728443  0.701269   
                5.000000  10.000000   0.555297  0.520194  0.565855  0.496835   
...                                        ...       ...       ...       ...   
Nigeria  Male   75.000000 80.000000   0.351996  0.298188  0.312676  0.250711   
                80.000000 85.000000   0.406368  0.438780  0.325427  0.407567   
                85.000000 90.000000   0.468550  0.372964  0.461027  0.401142   
                90.000000 95.000000   0.790239  0.790406  0.814604  0.779747   
                95.000000 100.000000  0.814702  0.794083  0.811262  0.809367   

                                      draw_101  draw_102  draw_103  draw_104  \
location sex    age_start age_end                                              
India    Female 0.000000  0.019178    0.890960  0.855054  0.900705  0.873191   
                0.019178  0.076712    0.594051  0.552846  0.589743  0.603153   
                0.076712  0.500000    0.655441  0.671472  0.655737  0.656435   
                0.500000  1.000000    0.727631  0.716115  0.744202  0.733556   
                5.000000  10.000000   0.557040  0.508719  0.506782  0.499247   
...                                        ...       ...       ...       ...   
Nigeria  Male   75.000000 80.000000   0.239969  0.222730  0.267228  0.218114   
                80.000000 85.000000   0.476539  0.406879  0.376987  0.267299   
                85.000000 90.000000   0.368977  0.479217  0.373618  0.430667   
                90.000000 95.000000   0.738822  0.810613  0.789167  0.804813   
                95.000000 100.000000  0.771474  0.824473  0.831639  0.738796   

                                      draw_105  draw_106  ...   draw_90  \
location sex    age_start age_end                         ...             
India    Female 0.000000  0.019178    0.882671  0.851349  ...  0.882201   
                0.019178  0.076712    0.583521  0.590471  ...  0.593110   
                0.076712  0.500000    0.632689  0.662699  ...  0.634974   
                0.500000  1.000000    0.676764  0.684945  ...  0.730368   
                5.000000  10.000000   0.548485  0.523623  ...  0.511438   
...                                        ...       ...  ...       ...   
Nigeria  Male   75.000000 80.000000   0.206974  0.305279  ...  0.381780   
                80.000000 85.000000   0.332065  0.260452  ...  0.412952   
                85.000000 90.000000   0.476602  0.398233  ...  0.550522   
                90.000000 95.000000   0.497839  0.720409  ...  0.842854   
                95.000000 100.000000  0.824819  0.860051  ...  0.872844   

                                       draw_91   draw_92   draw_93   draw_94  \
location sex    age_start age_end                                              
India    Female 0.000000  0.019178    0.886455  0.870279  0.898175  0.932924   
                0.019178  0.076712    0.550909  0.578389  0.587658  0.563501   
                0.076712  0.500000    0.650047  0.639402  0.655071  0.677751   
                0.500000  1.000000    0.729601  0.764601  0.733136  0.744315   
                5.000000  10.000000   0.521493  0.512802  0.551412  0.508867   
...                                        ...       ...       ...       ...   
Nigeria  Male   75.000000 80.000000   0.266314  0.413537  0.228288  0.222798   
                80.000000 85.000000   0.410859  0.566198  0.224144  0.318082   
                85.000000 90.000000   0.447799  0.432167  0.400423  0.408042   
                90.000000 95.000000   0.798896  0.837589  0.771650  0.802353   
                95.000

In [55]:
ira_prev.columns.name = "draw"
ira_prev = ira_prev.stack().pipe(lambda s: s[s.index.get_level_values("draw").isin(DRAWS)])
ira_prev

location  sex     age_start  age_end     draw    
India     Female  0.0        0.019178    draw_0      0.899755
                                         draw_1      0.857321
                                         draw_10     0.874569
                                         draw_100    0.887292
                                         draw_101    0.890960
                                                       ...   
Nigeria   Male    95.0       100.000000  draw_95     0.794225
                                         draw_96     0.814850
                                         draw_97     0.790197
                                         draw_98     0.853127
                                         draw_99     0.751525
Length: 46000, dtype: float64

In [56]:
iron_responsive_proportion = ira_prev / baseline_anemia.anemic.droplevel("vehicle")
iron_responsive_proportion

location  sex     age_start  age_end     draw     wealth_quintile
India     Female  0.0        0.019178    draw_0   lowest             0.911302
                                                  second             0.918687
                                                  middle             0.919568
                                                  fourth             0.931204
                                                  highest            0.947864
                                                                       ...   
Nigeria   Male    95.0       100.000000  draw_99  lowest             0.798873
                                                  second             0.810065
                                                  middle             0.814376
                                                  fourth             0.817734
                                                  highest            0.834934
Length: 230000, dtype: float64

In [57]:
# NOTE: I am pretty sure this is correct, but it is quite difficult to think through *why*.

# First, observe that people in the population who start as non-anemic never factor into
# any of these metrics. If they started non-anemic, our counterfactual can only shift them up,
# so they did not change anemia categories between scenarios and hence have no importance to
# anemia prevalence or YLDs.

# So you can think of our hemoglobin distributions as only being of interest in the part
# of them below the anemia threshold.

# *Within* this subpopulation, we make the assumption that iron-responsive and non-iron-responsive
# anemic people have the same distributions of hemoglobin. This is probably not true, but GBD doesn't
# give us anything better.

# So you can think of our original distribution as a mixture of two parts, which are the same.
# Then we shift one of those parts (the iron-responsive part) and calculate all these stats from
# that new distribution.

# Our *actual* result should be about a mixture distribution that has the non-iron-responsive part
# the same as in baseline, with the shifted iron-responsive part.
# For all these metrics, it is pretty straightforward to see that the metric in such a mixture is
# just a weighted average of the metrics in each part.

counterfactual_anemia_accounting_for_non_response = (
    counterfactual_anemia.mul(iron_responsive_proportion, axis=0) +
    baseline_anemia.mul(1 - iron_responsive_proportion, axis=0)
)
counterfactual_anemia_accounting_for_non_response

mild  \
location age_start age_end    sex    draw    wealth_quintile vehicle              
India    0.0       0.019178   Female draw_0  fourth          rice      0.434873   
                                             highest         rice      0.480059   
                                             lowest          rice      0.296196   
                                             middle          rice      0.377021   
                                             second          rice      0.369400   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 fourth          bouillon  0.312356   
                                             highest         bouillon  0.329780   
                                             lowest          bouillon  0.285928   
                                             middle          bouillon  0.308609   
                                             second          bouillon  0.303890   

                                                                       moderate  \
location age_start age_end    sex    draw    wealth_quintile vehicle              
India    0.0       0.019178   Female draw_0  fourth          rice      0.522801   
                                             highest         rice      0.455611   
                                             lowest          rice      0.685847   
                                             middle          rice      0.595177   
                                             second          rice      0.604107   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 fourth          bouillon  0.464933   
                                             highest         bouillon  0.445460   
                                             lowest          bouillon  0.487461   
                                             middle          bouillon  0.468562   
                                             second          bouillon  0.472894   

                                                                         severe  \
location age_start age_end    sex    draw    wealth_quintile vehicle              
India    0.0       0.019178   Female draw_0  fourth          rice      0.002214   
                                             highest         rice      0.001795   
                                             lowest          rice      0.003651   
                                             middle          rice      0.002760   
                                             second          rice      0.002836   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 fourth          bouillon  0.138701   
                                             highest         bouillon  0.122263   
                                             lowest          bouillon  0.162483   
                                             middle          bouillon  0.142127   
                                             second          bouillon  0.146407   

                                                                         anemic  \
location age_start age_end    sex    draw    wealth_quintile vehicle              
India    0.0       0.019178   Female draw_0  fourth          rice      0.959888   
                                             highest         rice      0.937465   
                                             lowest          rice      0.985694   
                                             middle          rice      0.974957   
                                             second          rice      0.976343   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 fourth          bouillon  0.915990   
                                             highest         bouillon  0.897503   
     

In [58]:
averted_anemia = baseline_anemia - counterfactual_anemia_accounting_for_non_response
averted_anemia

mild  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth          -0.019834   
                                                      highest         -0.019857   
                                                      lowest          -0.018806   
                                                      middle          -0.020325   
                                                      second          -0.018954   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth          -0.003410   
                                                      highest         -0.002127   
                                                      lowest          -0.008077   
                                                      middle          -0.004205   
                                                      second          -0.005863   

                                                                       moderate  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.025990   
                                                      highest          0.031450   
                                                      lowest           0.020199   
                                                      middle           0.023617   
                                                      second           0.021808   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.003330   
                                                      highest          0.002656   
                                                      lowest           0.005757   
                                                      middle           0.003910   
                                                      second           0.005125   

                                                                         severe  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.000183   
                                                      highest          0.000187   
                                                      lowest           0.000243   
                                                      middle           0.000205   
                                                      second           0.000194   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.003122   
                                                      highest          0.002068   
                                                      lowest           0.007178   
                                                      middle           0.003819   
                                                      second           0.005281   

                                                                         anemic  \
location age_start age_end    sex    draw    vehicle  wealth_quintile             
India    0.0       0.019178   Female draw_0  rice     fourth           0.006339   
                                                      highest          0.011780   
                                                      lowest           0.001635   
                                                      middle           0.003497   
                                                      second           0.003049   
...                                                                         ...   
Nigeria  95.0      100.000000 Male   draw_99 bouillon fourth           0.003043   
                                                      highest          0.002597   
     

In [59]:
assert (averted_anemia.anemic_ylds > 0).all()

In [60]:
pop = reformat_gbd_data(
    db_queries.get_population(
        location_id=list(location_crosswalk.location_id),
        age_group_id=age_group_ids,
        sex_id=list(sex_crosswalk.sex_id),
        year_id=2030,
        release_id=9,
        forecasted_pop=True
    )
).population
pop

location  sex     age_start  age_end   
India     Male    0.000000   0.019178      195123.148258
Nigeria   Male    0.000000   0.019178       78940.003165
India     Female  0.000000   0.019178      176190.386041
Nigeria   Female  0.000000   0.019178       76752.699967
India     Male    0.019178   0.076712      580497.144153
                                               ...      
Nigeria   Female  90.000000  95.000000      69350.824104
India     Male    95.000000  100.000000    269449.783462
Nigeria   Male    95.000000  100.000000     16933.452258
India     Female  95.000000  100.000000    371349.796191
Nigeria   Female  95.000000  100.000000     22274.971240
Name: population, Length: 84, dtype: float64

In [61]:
pop = reformat_gbd_data(db_queries.get_population(
    location_id=list(location_crosswalk.location_id),
    age_group_id=age_group_ids,
    sex_id=list(sex_crosswalk.sex_id),
    release_id=9,
    year_id=2021,
    forecasted_pop=False, # Was missing age groups!
)).population
pop

location  sex     age_start  age_end 
India     Male    0.000000   0.019178    2.183097e+05
Nigeria   Male    0.000000   0.019178    7.992764e+04
India     Female  0.000000   0.019178    1.975581e+05
Nigeria   Female  0.000000   0.019178    7.684611e+04
India     Male    0.019178   0.076712    6.478743e+05
                                             ...     
Nigeria   Female  0.076712   0.500000    1.653024e+06
India     Male    0.500000   1.000000    5.610299e+06
Nigeria   Male    0.500000   1.000000    1.966178e+06
India     Female  0.500000   1.000000    5.085507e+06
Nigeria   Female  0.500000   1.000000    1.904590e+06
Name: population, Length: 92, dtype: float64

In [62]:
import gbd_mapping

In [63]:
asfr = reformat_gbd_data(db_queries.get_covariate_estimates(
    int(gbd_mapping.covariates.age_specific_fertility_rate.gbd_id),
    location_id=list(location_crosswalk.location_id),
    age_group_id=age_group_ids,
    sex_id=list(sex_crosswalk.sex_id),
    year_id=2021,
    release_id=9,
).drop(columns=["location_name", "sex"])).mean_value
asfr

location  sex     age_start  age_end 
India     Male    0.000000   0.019178    0.0
Nigeria   Male    0.000000   0.019178    0.0
India     Female  0.000000   0.019178    0.0
Nigeria   Female  0.000000   0.019178    0.0
India     Male    0.019178   0.076712    0.0
                                        ... 
Nigeria   Female  0.076712   0.500000    0.0
India     Male    0.500000   1.000000    0.0
Nigeria   Male    0.500000   1.000000    0.0
India     Female  0.500000   1.000000    0.0
Nigeria   Female  0.500000   1.000000    0.0
Name: mean_value, Length: 92, dtype: float64

In [64]:
sbr = reformat_gbd_data(db_queries.get_covariate_estimates(
    int(gbd_mapping.covariates.stillbirth_to_live_birth_ratio.gbd_id),
    location_id=list(location_crosswalk.location_id),
    year_id=2021,
    release_id=9,
).drop(columns=["sex_id", "location_name", "sex", "age_group_id", "age_group_name"])).mean_value
sbr

location
India      0.017119
Nigeria    0.038684
Name: mean_value, dtype: float64

In [65]:
maternal_abortion_miscarriage_incidence = reformat_gbd_data(db_queries.get_outputs(
    topic='cause',
    release_id=9,
    year_id=2021,
    cause_id=int(gbd_mapping.causes.maternal_abortion_and_miscarriage.gbd_id),
    sex_id=sex_ids,
    location_id=list(location_crosswalk.location_id),
    age_group_id=age_group_ids,
    measure_id=6,
    metric_id=3,
).drop(columns=["sex", "location_name"])).val.fillna(0)
maternal_abortion_miscarriage_incidence.sort_values()

location  sex     age_start  age_end
India     Female  65.0       70.0       0.000000
Nigeria   Male    80.0       85.0       0.000000
India     Male    80.0       85.0       0.000000
Nigeria   Female  75.0       80.0       0.000000
India     Female  75.0       80.0       0.000000
                                          ...   
Nigeria   Female  20.0       25.0       0.035416
                  30.0       35.0       0.036698
India     Female  25.0       30.0       0.037986
Nigeria   Female  25.0       30.0       0.038699
India     Female  20.0       25.0       0.046091
Name: val, Length: 92, dtype: float64

In [66]:
ectopic_pregnancy_incidence = reformat_gbd_data(db_queries.get_outputs(
    topic='cause',
    release_id=9,
    year_id=2021,
    cause_id=int(gbd_mapping.causes.ectopic_pregnancy.gbd_id),
    sex_id=sex_ids,
    location_id=list(location_crosswalk.location_id),
    age_group_id=age_group_ids,
    measure_id=6,
    metric_id=3,
).drop(columns=["sex", "location_name"])).val.fillna(0)
ectopic_pregnancy_incidence.sort_values()

location  sex     age_start  age_end
India     Female  65.0       70.0       0.000000
Nigeria   Male    80.0       85.0       0.000000
India     Male    80.0       85.0       0.000000
Nigeria   Female  75.0       80.0       0.000000
India     Female  75.0       80.0       0.000000
                                          ...   
Nigeria   Female  20.0       25.0       0.008187
India     Female  25.0       30.0       0.008453
Nigeria   Female  35.0       40.0       0.009958
                  25.0       30.0       0.010354
                  30.0       35.0       0.012120
Name: val, Length: 92, dtype: float64

In [67]:
pregnancy_incidence = (
    asfr
    + (asfr * sbr)
    + maternal_abortion_miscarriage_incidence
    + ectopic_pregnancy_incidence
)
pregnancy_incidence[pregnancy_incidence > 0]

location  sex     age_start  age_end
India     Female  10.0       15.0       0.000395
Nigeria   Female  10.0       15.0       0.003561
India     Female  15.0       20.0       0.015350
Nigeria   Female  15.0       20.0       0.101501
India     Female  20.0       25.0       0.159986
Nigeria   Female  20.0       25.0       0.254413
India     Female  25.0       30.0       0.181117
Nigeria   Female  25.0       30.0       0.279034
India     Female  30.0       35.0       0.106541
Nigeria   Female  30.0       35.0       0.258724
India     Female  35.0       40.0       0.043143
Nigeria   Female  35.0       40.0       0.175022
India     Female  40.0       45.0       0.014067
Nigeria   Female  40.0       45.0       0.092714
India     Female  45.0       50.0       0.004355
Nigeria   Female  45.0       50.0       0.042147
India     Female  50.0       55.0       0.000471
Nigeria   Female  50.0       55.0       0.004797
dtype: float64

In [68]:
# https://github.com/ihmeuw/vivarium_gates_iv_iron/blob/862b057457cea2410e8d24c2c9c38c43419cd8b2/src/vivarium_gates_iv_iron/data/loader.py#L237C5-L240C6
PARTIAL_TERM: float = 24 / 52
FULL_TERM: float = 40 / 52
pregnant_prevalence = (
    FULL_TERM * (asfr + asfr * sbr)
    + PARTIAL_TERM * (maternal_abortion_miscarriage_incidence + ectopic_pregnancy_incidence)
)

In [69]:
pregnancy_pop = (pop * pregnant_prevalence)
pregnancy_pop[pregnancy_pop > 0]

location  sex     age_start  age_end
India     Female  10.0       15.0       1.892408e+04
Nigeria   Female  10.0       15.0       4.156882e+04
India     Female  15.0       20.0       6.499476e+05
Nigeria   Female  15.0       20.0       9.886794e+05
India     Female  20.0       25.0       6.821581e+06
Nigeria   Female  20.0       25.0       2.056488e+06
India     Female  25.0       30.0       7.486302e+06
Nigeria   Female  25.0       30.0       1.853251e+06
India     Female  30.0       35.0       4.196408e+06
Nigeria   Female  30.0       35.0       1.374654e+06
India     Female  35.0       40.0       1.583992e+06
Nigeria   Female  35.0       40.0       7.679469e+05
India     Female  40.0       45.0       4.451042e+05
Nigeria   Female  40.0       45.0       3.332663e+05
India     Female  45.0       50.0       1.170784e+05
Nigeria   Female  45.0       50.0       1.302636e+05
India     Female  50.0       55.0       1.046194e+04
Nigeria   Female  50.0       55.0       1.179362e+04
dtype: fl

In [70]:
non_pregnant_pop = pop - pregnancy_pop
non_pregnant_pop

location  sex     age_start  age_end 
India     Male    0.000000   0.019178    2.183097e+05
Nigeria   Male    0.000000   0.019178    7.992764e+04
India     Female  0.000000   0.019178    1.975581e+05
Nigeria   Female  0.000000   0.019178    7.684611e+04
India     Male    0.019178   0.076712    6.478743e+05
                                             ...     
Nigeria   Female  0.076712   0.500000    1.653024e+06
India     Male    0.500000   1.000000    5.610299e+06
Nigeria   Male    0.500000   1.000000    1.966178e+06
India     Female  0.500000   1.000000    5.085507e+06
Nigeria   Female  0.500000   1.000000    1.904590e+06
Length: 92, dtype: float64

In [71]:
baseline_ylds = (baseline_anemia.anemic_ylds.unstack("draw").mean(axis=1) * non_pregnant_pop)
baseline_ylds

location  age_start  age_end     sex     vehicle   wealth_quintile
India     0.0        0.019178    Female  rice      fourth             5750.771980
                                                   highest            5209.516075
                                                   lowest             7152.648727
                                                   middle             6367.337643
                                                   second             6430.516419
                                                                         ...     
Nigeria   95.0       100.000000  Male    bouillon  fourth              380.578395
                                                   highest             351.484752
                                                   lowest              424.345933
                                                   middle              387.245278
                                                   second              396.415810
Length: 460, dtype: float64

In [72]:
intervention_ylds = (counterfactual_anemia_accounting_for_non_response.anemic_ylds.unstack("draw").mean(axis=1) * non_pregnant_pop)

In [73]:
baseline_ylds.groupby(["location", "wealth_quintile"]).sum() - intervention_ylds.groupby(["location", "wealth_quintile"]).sum()

location  wealth_quintile
India     fourth             689946.027730
          highest            943640.988275
          lowest             501618.341505
          middle             613020.970442
          second             534975.129983
Nigeria   fourth              66259.360871
          highest             48541.842639
          lowest             132340.293560
          middle              79324.501100
          second             105829.837146
dtype: float64

In [74]:
ylds = pd.concat([
    baseline_ylds.rename("value").reset_index().assign(scenario="baseline"),
    intervention_ylds.rename("value").reset_index().assign(scenario="intervention")
], ignore_index=True)
ylds

,location,age_start,age_end,sex,vehicle,wealth_quintile,value,scenario
0,India,0.0,0.019178,Female,rice,fourth,5750.771980,baseline
1,India,0.0,0.019178,Female,rice,highest,5209.516075,baseline
2,India,0.0,0.019178,Female,rice,lowest,7152.648727,baseline
3,India,0.0,0.019178,Female,rice,middle,6367.337643,baseline
4,India,0.0,0.019178,Female,rice,second,6430.516419,baseline
...,...,...,...,...,...,...,...,...
915,Nigeria,95.0,100.000000,Male,bouillon,fourth,375.117253,intervention
916,Nigeria,95.0,100.000000,Male,bouillon,highest,347.674494,intervention
917,Nigeria,95.0,100.000000,Male,bouillon,lowest,412.586977,intervention
918,Nigeria,95.0,100.000000,Male,bouillon,middle,380.633678,intervention


In [75]:
import pathlib

for location in ylds.location.unique():
    path = f'./{location}/ylds.parquet'
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    ylds[ylds.location == location].to_parquet(path)

In [76]:
baseline_anemia_cases = baseline_anemia['anemic'].unstack("draw").mean(axis=1).mul(non_pregnant_pop, axis=0)
baseline_anemia_cases

location  age_start  age_end     sex     vehicle   wealth_quintile
India     0.0        0.019178    Female  rice      fourth             182072.755947
                                                   highest            176975.193249
                                                   lowest             190432.360172
                                                   middle             186483.358390
                                                   second             186863.573565
                                                                          ...      
Nigeria   95.0       100.000000  Male    bouillon  fourth               7843.840504
                                                   highest              7659.389484
                                                   lowest               8051.353400
                                                   middle               7880.280825
                                                   second               7927.317581
Length: 4

In [77]:
intervention_anemia_cases = counterfactual_anemia_accounting_for_non_response['anemic'].unstack("draw").mean(axis=1).mul(non_pregnant_pop, axis=0)
intervention_anemia_cases

location  age_start  age_end     sex     wealth_quintile  vehicle 
India     0.0        0.019178    Female  fourth           rice        180007.001360
                                         highest          rice        173703.048276
                                         lowest           rice        189635.357137
                                         middle           rice        185102.799651
                                         second           rice        185633.027977
                                                                          ...      
Nigeria   95.0       100.000000  Male    fourth           bouillon      7812.090096
                                         highest          bouillon      7631.800686
                                         lowest           bouillon      8000.752731
                                         middle           bouillon      7843.579597
                                         second           bouillon      7880.119843
Length: 4

In [78]:
(baseline_anemia_cases.groupby(["location", "wealth_quintile"]).sum() - intervention_anemia_cases.groupby(["location", "wealth_quintile"]).sum()).map(lambda x: f'{round(x):,.0f}')

location  wealth_quintile
India     fourth             18,091,890
          highest            25,756,279
          lowest             11,938,505
          middle             15,598,499
          second             13,374,540
Nigeria   fourth              1,622,614
          highest             1,311,831
          lowest              2,746,437
          middle              1,905,650
          second              2,418,965
dtype: object